In [1]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Informe NDVI: análisis de pérdidas/ganancias vs 2016\n",
        "\n",
        "Este notebook documenta y ejecuta el análisis que hemos estado realizando: limpieza de máscaras, vectorización previa (ya realizada), cálculo de áreas por municipio (GADM COL nivel 2), generación de reportes Top-10 por año, cálculo de net change y mapas coropléticos.\n",
        "\n",
        "Cómo usar:\n",
        "- Coloca este notebook en la raíz del proyecto (donde están las carpetas data/ y outputs/).\n",
        "- Asegúrate de que:\n",
        "  - outputs/ contiene `ndvi_loss_gain_vs_2016_cleaned.gpkg` y `areas_by_sector_vs_2016.csv` (generadas antes).\n",
        "  - data/ancillary/ contiene `gadm41_COL.gpkg`.\n",
        "- Ejecuta las celdas en orden. Ajusta parámetros en la celda \"Parámetros\" si lo necesitas.\n",
        "\n",
        "Requisitos de Python (instala en tu env): geopandas, fiona, rasterio, shapely, pandas, matplotlib, seaborn, folium, pyproj, scikit-image (si regenera máscaras), pytest (opcional).\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: Parámetros y imports\n",
        "import os\n",
        "from glob import glob\n",
        "import pandas as pd\n",
        "import geopandas as gpd\n",
        "import fiona\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "import folium\n",
        "\n",
        "# Ajusta según tu entorno\n",
        "BASE = os.getcwd()\n",
        "RAW = os.path.join(BASE, 'data', 'raw')\n",
        "ANC = os.path.join(BASE, 'data', 'ancillary')\n",
        "OUT = os.path.join(BASE, 'outputs')\n",
        "os.makedirs(OUT, exist_ok=True)\n",
        "\n",
        "# Archivos esperados\n",
        "GPKG_MASKS = os.path.join(OUT, 'ndvi_loss_gain_vs_2016_cleaned.gpkg')\n",
        "CSV_AREAS = os.path.join(OUT, 'areas_by_sector_vs_2016.csv')\n",
        "GADM_GPKG = os.path.join(ANC, 'gadm41_COL.gpkg')\n",
        "\n",
        "# Parámetros de análisis\n",
        "ID_FIELD = 'NAME_2'   # campo identificador en GADM (ajusta si es otro)\n",
        "TARGET_CRS = 'EPSG:32618'  # CRS métrico para cálculo de áreas (UTM 18N)\n",
        "TOPN = 10\n",
        "SNS_PALETTE_GAIN = 'Greens'\n",
        "SNS_PALETTE_LOSS = 'Reds'\n",
        "\n",
        "print('BASE:', BASE)\n",
        "print('RAW exists:', os.path.exists(RAW))\n",
        "print('ANC exists:', os.path.exists(ANC))\n",
        "print('OUT exists:', os.path.exists(OUT))\n",
        "print('Masks GPKG:', os.path.exists(GPKG_MASKS), GPKG_MASKS)\n",
        "print('CSV areas exists:', os.path.exists(CSV_AREAS), CSV_AREAS)\n",
        "print('GADM GPKG exists:', os.path.exists(GADM_GPKG), GADM_GPKG)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: inspeccionar GADM (capa ADM_ADM_2) y CSV de áreas (resumen previo)\n",
        "if not os.path.exists(GADM_GPKG):\n",
        "    raise SystemExit('No se encuentra GADM en data/ancillary. Coloca gadm41_COL.gpkg en data/ancillary.')\n",
        "layers = fiona.listlayers(GADM_GPKG)\n",
        "print('GADM layers:', layers)\n",
        "layer_gadm = None\n",
        "for L in layers:\n",
        "    if L.endswith('_2') or 'ADM_ADM_2' in L:\n",
        "        layer_gadm = L\n",
        "        break\n",
        "if layer_gadm is None:\n",
        "    layer_gadm = layers[0]\n",
        "print('Using GADM layer:', layer_gadm)\n",
        "gadm = gpd.read_file(GADM_GPKG, layer=layer_gadm)\n",
        "print('GADM CRS:', gadm.crs, 'features:', len(gadm))\n",
        "print('Columns:', list(gadm.columns))\n",
        "display(gadm.head())\n",
        "\n",
        "if not os.path.exists(CSV_AREAS):\n",
        "    print('Warning: CSV with areas not found at', CSV_AREAS)\n",
        "else:\n",
        "    df_areas = pd.read_csv(CSV_AREAS)\n",
        "    print('Loaded areas CSV rows:', len(df_areas))\n",
        "    display(df_areas.head())\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: Generar informe Top-N por año (CSV + PNGs) — crea outputs/top10_by_year.csv y PNGs\n",
        "if not os.path.exists(CSV_AREAS):\n",
        "    raise SystemExit('El CSV areas_by_sector_vs_2016.csv no existe. Ejecuta la celda que lo genera primero.')\n",
        "\n",
        "df = pd.read_csv(CSV_AREAS)\n",
        "df.columns = [c.strip() for c in df.columns]\n",
        "df['year'] = df['year'].astype(int)\n",
        "df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce').fillna(0)\n",
        "\n",
        "rows = []\n",
        "sns.set_style('whitegrid')\n",
        "for y in sorted(df['year'].unique()):\n",
        "    for kind in ['gain','loss']:\n",
        "        sub = df[(df['year']==y) & (df['kind']==kind)].copy()\n",
        "        if sub.empty:\n",
        "            continue\n",
        "        sub_sorted = sub.sort_values('area_ha', ascending=False).head(TOPN).reset_index(drop=True)\n",
        "        sub_sorted['rank'] = sub_sorted.index + 1\n",
        "        sub_sorted['kind'] = kind\n",
        "        sub_sorted['year'] = y\n",
        "        rows.append(sub_sorted[[ID_FIELD,'area_ha','year','kind','rank']])\n",
        "\n",
        "        # plot\n",
        "        plt.figure(figsize=(8,6))\n",
        "        pal = SNS_PALETTE_GAIN if kind=='gain' else SNS_PALETTE_LOSS\n",
        "        sns.barplot(x='area_ha', y=ID_FIELD, data=sub_sorted, palette=pal)\n",
        "        plt.xlabel('Área (ha)')\n",
        "        plt.title(f'Top {TOPN} {kind.upper()} — {y}')\n",
        "        plt.tight_layout()\n",
        "        out_png = os.path.join(OUT, f'top{TOPN}_{kind}_{y}.png')\n",
        "        plt.savefig(out_png, dpi=150, bbox_inches='tight')\n",
        "        plt.close()\n",
        "        print('Saved:', out_png)\n",
        "\n",
        "if rows:\n",
        "    df_top = pd.concat(rows, ignore_index=True)\n",
        "    csv_out = os.path.join(OUT, 'top10_by_year.csv')\n",
        "    df_top.to_csv(csv_out, index=False)\n",
        "    print('Saved summary CSV:', csv_out)\n",
        "else:\n",
        "    print('No data to create Top-N report')\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: Calcular net change por municipio por año, acumulado y porcentaje afectado\n",
        "df = pd.read_csv(CSV_AREAS)\n",
        "df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce').fillna(0)\n",
        "pivot = df.pivot_table(index=['NAME_2','year'], columns='kind', values='area_ha', aggfunc='sum').fillna(0).reset_index()\n",
        "pivot['net_ha'] = pivot.get('gain',0) - pivot.get('loss',0)\n",
        "\n",
        "# acumulado por municipio (suma net al final del periodo)\n",
        "acc = pivot.groupby('NAME_2').apply(lambda g: g.sort_values('year')['net_ha'].cumsum().iloc[-1]).reset_index()\n",
        "acc.columns = ['NAME_2','cumulative_net_ha']\n",
        "\n",
        "# unir con sectores para obtener area total\n",
        "sectors = gadm[[ 'NAME_2', 'geometry' ]].copy()\n",
        "sectors = sectors.to_crs(TARGET_CRS)\n",
        "sectors['area_ha_total'] = sectors.geometry.area / 10000.0\n",
        "\n",
        "acc = acc.merge(sectors[['NAME_2','area_ha_total']], on='NAME_2', how='left')\n",
        "acc['pct_affected'] = 100.0 * acc['cumulative_net_ha'] / acc['area_ha_total']\n",
        "acc = acc.fillna(0)\n",
        "acc_out = os.path.join(OUT, 'municipios_cumulative_net_pct.csv')\n",
        "acc.to_csv(acc_out, index=False)\n",
        "print('Saved cumulative file:', acc_out)\n",
        "display(acc.sort_values('pct_affected', ascending=False).head(20))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: Exportar GeoJSON/GPKG con net change para 2025 (o cualquier año)\n",
        "year_to_export = 2025\n",
        "# recompute pivot in case\n",
        "df = pd.read_csv(CSV_AREAS)\n",
        "df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce').fillna(0)\n",
        "pivot = df.pivot_table(index=['NAME_2','year'], columns='kind', values='area_ha', aggfunc='sum').fillna(0).reset_index()\n",
        "pivot['net_ha'] = pivot.get('gain',0) - pivot.get('loss',0)\n",
        "net_yr = pivot[pivot['year']==year_to_export][['NAME_2','net_ha']].copy()\n",
        "\n",
        "gdf_out = sectors.to_crs('EPSG:32618').merge(net_yr, on='NAME_2', how='left').fillna(0)\n",
        "gdf_out['net_ha'] = gdf_out['net_ha'].fillna(0)\n",
        "gpkg_out = os.path.join(OUT, f'areas_by_municipio_{year_to_export}.gpkg')\n",
        "geojson_out = os.path.join(OUT, f'areas_by_municipio_{year_to_export}.geojson')\n",
        "gdf_out.to_file(gpkg_out, layer=f'net_{year_to_export}', driver='GPKG')\n",
        "gdf_out.to_file(geojson_out, driver='GeoJSON')\n",
        "print('Saved:', gpkg_out, geojson_out)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# CELDA: Crear mapa coroplético interactivo (net change % acumulado) y guardarlo\n",
        "gdf_pct = acc.merge(sectors[['NAME_2','geometry']], on='NAME_2', how='left').fillna(0)\n",
        "gdf_pct = gdf_pct.set_geometry('geometry').set_crs(TARGET_CRS)\n",
        "gdf_pct_4326 = gdf_pct.to_crs(epsg=4326)\n",
        "\n",
        "m = folium.Map(location=[11.0, -74.85], zoom_start=10, tiles='CartoDB positron')\n",
        "chor = folium.Choropleth(\n",
        "    geo_data=gdf_pct_4326.__geo_interface__,\n",
        "    data=gdf_pct_4326,\n",
        "    columns=['NAME_2','pct_affected'],\n",
        "    key_on='feature.properties.NAME_2',\n",
        "    fill_color='YlOrRd',\n",
        "    fill_opacity=0.7,\n",
        "    line_opacity=0.2,\n",
        "    legend_name='Pct affected (cumulative net %)' \n",
        ").add_to(m)\n",
        "\n",
        "folium.GeoJson(gdf_pct_4326.__geo_interface__,\n",
        "               name='details',\n",
        "               tooltip=folium.features.GeoJsonTooltip(fields=['NAME_2','pct_affected'],\n",
        "                                                      aliases=['Municipio','Pct affected (%)'],\n",
        "                                                      localize=True)\n",
        "              ).add_to(m)\n",
        "\n",
        "out_html = os.path.join(OUT, 'choropleth_cumulative_pct.html')\n",
        "m.save(out_html)\n",
        "print('Saved map:', out_html)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "Conclusión y siguientes pasos:\n",
        "- Revisa las figuras en outputs/ (top10 PNGs) y el CSV top10_by_year.csv. Visualiza choropleth_cumulative_pct.html.\n",
        "- Validación visual: abre outputs/areas_by_municipio_2025.geojson en QGIS y compara con overlays (outputs/overlay_*.png) y el NDVI base.\n",
        "- Si ves falsos positivos, reejecuta la limpieza de máscaras ajustando min_area_ha y vuelve a vectorizar antes de repetir este notebook.\n",
        "\n",
        "Si quieres, puedo:\n",
        "1) ajustar el umbral min_area_ha y regenerar máscaras + vectores (necesito acceso al código de generación de máscaras que usaste),\n",
        "2) preparar un PDF resumen con tablas y gráficos, o\n",
        "3) exportar los resultados finales a un GeoPackage listo para entrega.\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# Informe NDVI: análisis de pérdidas/ganancias vs 2016\n',
    '\n',
    'Este notebook documenta y ejecuta el análisis que hemos estado realizando: limpieza de máscaras, vectorización previa (ya realizada), cálculo de áreas por municipio (GADM COL nivel 2), generación de reportes Top-10 por año, cálculo de net change y mapas coropléticos.\n',
    '\n',
    'Cómo usar:\n',
    '- Coloca este notebook en la raíz del proyecto (donde están las carpetas data/ y outputs/).\n',
    '- Asegúrate de que:\n',
    '  - outputs/ contiene `ndvi_loss_gain_vs_2016_cleaned.gpkg` y `areas_by_sector_vs_2016.csv` (generadas antes).\n',
    '  - data/ancillary/ contiene `gadm41_COL.gpkg`.\n',
    '- Ejecuta las celdas en orden. Ajusta parámetros en la celda "Parámetros" si lo necesitas.\n',
    '\n',
    'Requisitos de Python (instala en tu env): geopandas, fiona, rasterio, shapely, pandas, matplotlib, seaborn, folium, pyproj, scikit-